# Exporing the data exported frm the Fitness app

This approach uses data that has been processes from the raw csv files, which are large.  Those files are available in *HealthAll...* directory under book_data.  The spark processes *read_<itme>_exports.py* drop the parquet files in *book_data/sparkOutput*.

## The usual library imports and location setup

In [1]:
#import matplotlib.pyplot as plt
import pandas as pd
import numpy as np
#from sklearn.linear_model import LinearRegression
#from sklearn.model_selection import train_test_split
#from sklearn.metrics import mean_squared_error
from pathlib import Path
from os import path, listdir
import re
from datetime import datetime,tzinfo,date
from dateutil import parser
import pytz

# Set up for locations
home = Path(path.abspath('..'))
user_home = Path('/Users/edmundlskoviak/')

# book_data is a legacy from the repo I built upon 
data = home / 'book_data'
# Files down loaded from HeathKit
#fitness_data = user_home / 'iCloud' / 'Data Sets' /'HealthAll_2024-12-345_14-58-25_SimpleHealthExportCSV'

In [19]:
# Attempt to read a parquet file

# Activity
activity_file = data / 'sparkOutput' / 'activity'
activity_df = pd.read_parquet(activity_file)

# Quantities
quantity_file = data / 'sparkOutput' / 'quantity'
quantity_df = pd.read_parquet(quantity_file)


# Fix the start date and end date
activity_df['startDate'] = activity_df.apply(lambda row: pd.Timestamp(row['startDate'], tz=pytz.UTC), axis=1)
activity_df['endDate'] = activity_df.apply(lambda row: pd.Timestamp(row['endDate'], tz=pytz.UTC), axis=1)

quantity_df['startDate'] = quantity_df.apply(lambda row: pd.Timestamp(row['startDate'], tz=pytz.UTC), axis=1)
quantity_df['endDate'] = quantity_df.apply(lambda row: pd.Timestamp(row['endDate'], tz=pytz.UTC), axis=1)

# Fix the cal qty to ba a float
quantity_df['value']=quantity_df['value'].astype(float)


# This is used to add a single day to a pd.TimeStamp data type
one_day = pd.Timedelta(1, 'd')



start_date = pd.Timestamp('2024-09-02', tz=pytz.UTC)
end_date = start_date + one_day



day_df = activity_df[(activity_df['startDate'] >= start_date) &
             (activity_df['startDate'] < end_date)]
total_activity = quantity_df[(quantity_df['startDate'] >= start_date )& (
                        quantity_df['startDate'] <= end_date)]['value'].sum()

# 5 = startDate
# 6 = endData
# 7 = activity Name
print(f'Date: {start_date}\nTotal Activity: {np.round(total_activity,0)}')
for row in day_df.itertuples(index=False):
    #print(row[5], row[6], row[7])
    filtered_df = quantity_df[(quantity_df['startDate'] >= row[5]) &
                           (quantity_df['startDate'] <= row[6])]
    print(f'{row[7]}: {np.round(filtered_df['value'].sum(),0)}')                       


Date: 2024-09-02 00:00:00+00:00
Total Activity: 1158.0
TraditionalStrengthTraining: 422.0
Walking: 146.0
Walking: 11.0
Walking: 149.0


In [34]:
# Read Activity File



# Fix the start date and end date


# Fix the value to be a float


activity_start = pd.Timestamp('2024-09-02 00:00:00+00:00', tz=pytz.UTC)
activity_end = pd.Timestamp('2024-09-03 00:00:00+00:00', tz=pytz.UTC)





In [21]:
activity_sum = filtered_df['value'].sum()
print(np.round(activity_sum,1))
#plt.hist(filtered_df['value'], bins=10)





1157.7


In [36]:
one_day = pd.Timedelta(1, 'd')
some_date = pd.Timestamp('2024-01-05 00:00:00+00:00')
another_date = some_date + day

print(some_date, another_date)

2024-01-05 00:00:00+00:00 2024-01-06 00:00:00+00:00


In [17]:
day = pd.Interval(start_date, end_date)
#pd.Timestamp('2024-09-03T12:00:00', tz=pytz.UTC) in day
quantity_df[quantity_df['startDate'] in day]

ValueError: The truth value of a Series is ambiguous. Use a.empty, a.bool(), a.item(), a.any() or a.all().

DEPRECATED

In [ ]:
# Test the fitness_data path
#json_files = [f for f in listdir(json_path) if path.isfile(path.join(json_path, f)) and f.split('.')[1] == 'json']
def convertDate(d : str) -> datetime:
    return parser.parse(d)

item = 'FlightsClimbed'
HKQuantity = re.compile(r'^HKQuantityTypeIdentifier'+item)
csv_files = [f for f in listdir(fitness_data) if path.isfile(path.join(fitness_data, f)) and f.split('.')[1] == 'csv' and HKQuantity.match(f)]
#dtypes = {"startDate": datetime64[ns, tzutc()], "endDate":datetime64[ns, tzutc()]}
converters={"startDate" : convertDate, "endDate" : convertDate}
columns = ['type','sourceVersion','productType','startDate','endDate','unit','value']
HKQuantity_df = pd.read_csv(fitness_data / csv_files[0], skiprows=0, header=1, usecols=columns, converters=converters)


HKQuantity_filtered_df = HKQuantity_df[HKQuantity_df['startDate'] < pd.Timestamp('2024-09-02',tz=pytz.UTC)]
print(f'{item}: {HKQuantity_filtered_df['value'].sum()}')

## Lets look at some activity

Filename starts with 'HKWorkoutActivityType<>'

Where <> in

* CrossTraining
* Cycling
* Eliptical
* FunctionalStrengthTraining
* Golf
* Hiking
* Other
* PaddleSports
* Running
* Snowboarding
* SnowSports
* Swimming
* TraditionalStrengthTraining (does not have HKElevationAscent)
* Walking (has HKElevationAscent), also has HKIndoorWorkout flag)
* WaterSports
* Yoga

for 9/1-

* 879/800 Cal (Move)
* 104/90 Min 
* 16/12 hrs

* 16728 steps
* 7.30 Mi
* 6 flights climbed

Outdoor walk 391 active cal

In [ ]:
def convertDate(d : str) -> datetime:
    return parser.parse(d)

item = 'Yoga'
HKActivity = re.compile(r'^HKWorkoutActivityType'+item)
csv_files = [f for f in listdir(fitness_data) if path.isfile(path.join(fitness_data, f)) and f.split('.')[1] == 'csv' and HKActivity.match(f)]
#dtypes = {"startDate": datetime64[ns, tzutc()], "endDate":datetime64[ns, tzutc()]}
converters={"startDate" : convertDate, "endDate" : convertDate}
columns = ['type','sourceVersion','productType','startDate','endDate','activityType','duration','durationUnit','totalEnergyBurned','totalDistance',
           'totalSwimmingStrokeCount', 'totalFlightsClimbed','HKIndoorWorkout','HKWeatherHumidity','HKTimeZone','HKAverageMETs',
           #'HKElevationAscended',
           'HKWeatherTemperature']
HKActivity_df = pd.read_csv(fitness_data / csv_files[0], skiprows=0, header=1, usecols=columns, converters=converters)

HKActivity_filtered_df = HKActivity_df[HKActivity_df['startDate'] < pd.Timestamp('2024-11-16',tz=pytz.UTC)]
print(f'{item}: {HKActivity_filtered_df['totalEnergyBurned'].sum()}')
